# **Abstract Syntax Tree**

In [23]:
import ast

In [24]:
eight = ast.Constant(8)

In [25]:
neg_eight = ast.UnaryOp(ast.USub(), eight)

In [26]:
read = ast.Call(ast.Name('input_int'),[])

In [27]:
simple_ast = ast.BinOp(read, ast.Add(), neg_eight)

In [28]:
ast_test_code = """
print(5+6)
20 - input_int()
"""
ast_parsed = ast.parse(ast_test_code)

In [29]:
print(ast.dump(ast_parsed,indent=4))

Module(
    body=[
        Expr(
            value=Call(
                func=Name(id='print', ctx=Load()),
                args=[
                    BinOp(
                        left=Constant(value=5),
                        op=Add(),
                        right=Constant(value=6))])),
        Expr(
            value=BinOp(
                left=Constant(value=20),
                op=Sub(),
                right=Call(
                    func=Name(id='input_int', ctx=Load()))))])


In [30]:
match simple_ast:
    case ast.BinOp(child1 , op, child2):
        print(op)

Add()


In [31]:
def leaf(arith):
    match arith:
        case ast.Constant(n):
            return True
        case ast.Call(ast.Name('input_int'),[]):
            return True
        case ast.UnaryOp(ast.USub(),e1):
            return False
        case ast.BinOp(e1,ast.Add(),e2):
            return False
        case ast.BinOp(e1,ast.Sub(),e2):
            return False
        case _:
            raise Exception('unexpected node')

In [32]:
print(leaf(ast.Call(ast.Name('input_int'),[])))

True


In [33]:
print(leaf(ast.UnaryOp(ast.USub(),eight)))

False


In [34]:
print(leaf(ast.Constant(eight)))

True


In [37]:
try:
    print(leaf(ast.BinOp(eight, ast.Mult(), eight)))
except Exception:
    print('error! unexpected node')

error! unexpected node


In [38]:
def is_exp(e):
    match e:
        case ast.Constant(n):
            return True
        case ast.Call(ast.Name('input_int'),[]):
            return True
        case ast.UnaryOp(ast.USub(),e1):
            return is_exp(e1)
        case ast.BinOp(e1, ast.Add(), e2):
            return is_exp(e1) and is_exp(e2)
        case ast.BinOp(e1,ast.Sub(), e2):
            return is_exp(e1) and is_exp(e2)
        case _:
            return False

In [39]:
def is_stmt(s):
    match s:
        case ast.Expr(ast.Call(ast.Name('input_int'),[e])):
            return is_exp(e)
        case ast.Expr(e):
            return is_exp(e)
        case _:
            return False

In [40]:
def is_Lint(p):
    match p:
        case ast.Module(body):
            return all([is_stmt(s) for s in body])
        case _:
            return False

In [43]:
print(is_Lint(ast.Module([ast.Expr(simple_ast)])))

True


In [45]:
print(is_Lint(ast.Module([ast.Expr(ast.BinOp(read,ast.Sub(),ast.UnaryOp(ast.Add(),eight)))])))

False
